# Does CDVAE beat a simple loop? A fair test on Perov-5

Run the cells from top to bottom on a Google Colab T4 GPU.
Cells 1-4 set everything up (about 25 minutes the first time), Cell 5 is a 3-minute test,
Cell 6 trains, Cell 7 generates, Cells 8-9 score the results.
Result: the simple loop wins. See the README for the numbers.


Cell 1: get the code

In [1]:
![ -d cdvae ] || git clone -q https://github.com/txie-93/cdvae

Cell 2: fair split (about 1 min)

In [2]:
"""Step 1 - audit Perov-5 and build a fair train/val/test split (plus menu.csv for later steps)."""
import collections, csv, itertools, os, random, re
import numpy as np

DATA = "cdvae/data/perov_5"            # where CDVAE keeps the original files
OUT = "cdvae/data/perov_5_grouped"     # where the fair split will be written
SEED = 42
AVOID = "Co"                           # element used for the "how much does avoiding it remove?" check

csv.field_size_limit(10**9)
ANIONS = {"O", "N", "F", "S"}
IDEAL = np.array([[0, 0, 0], [.5, .5, .5], [.5, .5, 0], [.5, 0, .5], [0, .5, .5]])  # ideal cubic ABX3


def read_cif(cif):
    """Return the cubic cell length a (Angstrom) and a list of (element, fractional xyz)."""
    a = float(re.search(r"_cell_length_a\s+([\d.]+)", cif).group(1))
    lines = cif.splitlines()
    start = max(i for i, l in enumerate(lines) if "_atom_site_occupancy" in l) + 1
    atoms = []
    for l in lines[start:]:
        t = l.split()
        if len(t) >= 7:
            atoms.append((t[0], np.array([float(t[3]), float(t[4]), float(t[5])])))
    return a, atoms


def dist(p, q, a):
    """Shortest distance between two points in the periodic cube (Angstrom)."""
    d = p - q
    return np.linalg.norm((d - np.round(d)) * a)


IDEAL_D = np.sort([dist(IDEAL[i], IDEAL[j], 1.0) for i, j in itertools.combinations(range(5), 2)])


def describe(row):
    a, atoms = read_cif(row["cif"])
    anions = [(s, p) for s, p in atoms if s in ANIONS]
    cations = [(s, p) for s, p in atoms if s not in ANIONS]
    assert len(atoms) == 5 and len(anions) == 3 and len(cations) == 2
    # The B site sits closest to the anions (octahedron); the other metal is the A site.
    near = [min(dist(p, q, a) for _, q in anions) for _, p in cations]
    b = int(np.argmin(near))
    A, B = cations[1 - b][0], cations[b][0]
    X = "".join(sorted(s for s, _ in anions))
    pos = [p for _, p in atoms]
    d = np.sort([dist(pos[i], pos[j], a) for i, j in itertools.combinations(range(5), 2)]) / a
    return dict(A=A, B=B, anions=X, composition=(tuple(sorted([A, B])), X),
                distortion=float(np.max(np.abs(d - IDEAL_D))))


rows, fields = [], None
for split in ["train", "val", "test"]:
    with open(os.path.join(DATA, split + ".csv"), newline="") as f:
        reader = csv.DictReader(f)
        fields = reader.fieldnames
        for row in reader:
            row["_split"] = split
            row.update(describe(row))
            rows.append(row)
n = len(rows)
sizes = [sum(r["_split"] == s for r in rows) for s in ["train", "val", "test"]]
print(f"structures: {n}  (train/val/test = {sizes})")

# 1. The menu check
metals = {r["A"] for r in rows} | {r["B"] for r in rows}
anion_sets = sorted({r["anions"] for r in rows})
combos = {(r["A"], r["B"], r["anions"]) for r in rows}
grid = len(metals) ** 2 * len(anion_sets)
print(f"metals: {len(metals)} | anion sets: {len(anion_sets)} {anion_sets}")
print(f"full grid = {grid} | structures = {n} | each combination listed exactly once: {len(combos) == n == grid}")

# 2. How much the original split leaks
train_comps = {r["composition"] for r in rows if r["_split"] == "train"}
for s in ["val", "test"]:
    sub = [r for r in rows if r["_split"] == s]
    hit = sum(r["composition"] in train_comps for r in sub)
    print(f"original split: {hit}/{len(sub)} {s} structures ({100 * hit / len(sub):.1f}%) share a composition with train")

# 3. How much does avoiding one element remove?
k = sum(AVOID in (r["A"], r["B"]) for r in rows)
print(f"structures containing {AVOID}: {k} ({100 * k / n:.1f}% of the data)")

# 4. Atoms moved away from the ideal cube?
dis = np.array([r["distortion"] for r in rows])
print(f"near the ideal cube: {100 * (dis < 0.02).mean():.1f}% | relaxed away: {100 * (dis >= 0.02).mean():.1f}% "
      f"| far away: {100 * (dis >= 0.10).mean():.1f}%")

# 5. Fair split: keep every composition (both of its polymorphs) in ONE set only
groups = collections.defaultdict(list)
for r in rows:
    groups[r["composition"]].append(r)
keys = sorted(groups)
random.Random(SEED).shuffle(keys)
new, count = {"train": [], "val": [], "test": []}, 0
for key in keys:
    name = "train" if count < 0.6 * n else "val" if count < 0.8 * n else "test"
    new[name] += groups[key]
    count += len(groups[key])

os.makedirs(OUT, exist_ok=True)
for name, items in new.items():
    with open(os.path.join(OUT, name + ".csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in items:
            w.writerow({c: r[c] for c in fields})
comps = {name: {r["composition"] for r in items} for name, items in new.items()}
assert not (comps["train"] & comps["val"] or comps["train"] & comps["test"] or comps["val"] & comps["test"])
print(f"new split: { {name: len(items) for name, items in new.items()} } | compositions shared between sets: 0")

# 6. menu.csv: one line per combination, used by steps 2 and 4
fair = {r["material_id"]: name for name, items in new.items() for r in items}
with open(os.path.join(OUT, "menu.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["material_id", "A", "B", "anions", "heat_ref", "split_original", "split_fair", "distortion"])
    for r in rows:
        w.writerow([r["material_id"], r["A"], r["B"], r["anions"], r["heat_ref"], r["_split"],
                    fair[r["material_id"]], f"{r['distortion']:.4f}"])
print(f"written to {OUT}  (train.csv, val.csv, test.csv, menu.csv)")

structures: 18928  (train/val/test = [11356, 3787, 3785])
metals: 52 | anion sets: 7 ['FNO', 'FOO', 'NNN', 'NNO', 'NOO', 'OOO', 'OOS']
full grid = 18928 | structures = 18928 | each combination listed exactly once: True
original split: 2265/3787 val structures (59.8%) share a composition with train
original split: 2214/3785 test structures (58.5%) share a composition with train
structures containing Co: 721 (3.8% of the data)
near the ideal cube: 57.8% | relaxed away: 42.2% | far away: 36.5%
new split: {'train': 11358, 'val': 3785, 'test': 3785} | compositions shared between sets: 0
written to cdvae/data/perov_5_grouped  (train.csv, val.csv, test.csv, menu.csv)


Cell 3: Setup (Drive, settings, speed patch)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RUNS = "/content/drive/MyDrive/cdvae_runs"
CACHE = "/content/drive/MyDrive/cdvae_cache"
PY = "/content/cdvae_env/bin/python"
ENV = f"PROJECT_ROOT=/content/cdvae HYDRA_JOBS={RUNS} WABDB_DIR=/content/wandb CDVAE_CACHE={CACHE}"
COMMON = ("data=perov model.predict_property=True data.root_path=/content/cdvae/data/perov_5_grouped "
          "data.preprocess_workers=2 logging.wandb.mode=offline")
for d in ["/content/wandb", RUNS, CACHE]:
    os.makedirs(d, exist_ok=True)

# one-time patch: remember the slow data conversion on Drive instead of redoing it at every start
p = "/content/cdvae/cdvae/pl_data/dataset.py"
s = open(p).read()
if "_cached_preprocess" in s:
    print("patch already applied")
else:
    helper = '''
import hashlib, os, pickle


def _cached_preprocess(path, num_workers, **kwargs):
    """preprocess(), but the result is remembered in the folder named by the env variable CDVAE_CACHE."""
    cache_dir = os.environ.get("CDVAE_CACHE")
    if not cache_dir:
        return preprocess(path, num_workers, **kwargs)
    with open(path, "rb") as f:
        digest = hashlib.md5(f.read()).hexdigest()
    settings = repr(sorted((k, str(v)) for k, v in kwargs.items()))
    key = hashlib.md5((digest + settings).encode()).hexdigest()[:16]
    cache_file = os.path.join(cache_dir, os.path.basename(path).replace(".csv", "") + "_" + key + ".pkl")
    if os.path.exists(cache_file):
        print("[cache] loading converted data from", cache_file, flush=True)
        with open(cache_file, "rb") as f:
            return pickle.load(f)
    out = preprocess(path, num_workers, **kwargs)
    os.makedirs(cache_dir, exist_ok=True)
    with open(cache_file + ".part", "wb") as f:
        pickle.dump(out, f)
    os.replace(cache_file + ".part", cache_file)
    print("[cache] saved converted data to", cache_file, flush=True)
    return out

'''
    s = s.replace("\n\nclass CrystDataset(Dataset):", helper + "\nclass CrystDataset(Dataset):", 1)
    s = s.replace("self.cached_data = preprocess(", "self.cached_data = _cached_preprocess(", 1)
    open(p, "w").write(s)
    print("patch applied")

Mounted at /content/drive
patch applied


Cell 4: Environment (builds, or restores from Drive, then saves the backup)

In [ ]:
%%bash
set -e
ROOT=${ROOT:-/content}
DRIVE=${DRIVE:-/content/drive/MyDrive}
export CONDA_ALWAYS_YES=true MAMBA_ALWAYS_YES=true PIP_DISABLE_PIP_VERSION_CHECK=1
MAMBA=$ROOT/mf/bin/mamba
CENV=$ROOT/cdvae_env
PIP=$CENV/bin/pip
PY=$CENV/bin/python
CH="--override-channels -c pytorch -c conda-forge"
ARCHIVE=$DRIVE/cdvae_env_backup.tar.gz
cd $ROOT
[ -d $DRIVE ] || { echo "STOP: Google Drive is not mounted. Run the Setup cell first, then run this cell again."; exit 1; }

echo "== 0. repo (skipped if already there) =="
[ -d $ROOT/cdvae ] || git clone -q https://github.com/txie-93/cdvae $ROOT/cdvae

echo "== 1. restore the environment saved on Drive, if there is one =="
if [ ! -x $PY ] && [ -f $ARCHIVE ]; then
  echo "restoring $ARCHIVE (a few minutes) ..."
  tar -xzf $ARCHIVE -C $(dirname $ROOT)
fi

if $PY -c "import torch, cdvae" 2>/dev/null; then
  echo "== 2. environment is already complete: skipping all installs =="
else
  echo "== 2. building the environment (about 20-25 minutes) =="
  if [ ! -x $MAMBA ]; then
    wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O $ROOT/mf.sh
    bash $ROOT/mf.sh -b -p $ROOT/mf > /dev/null
  fi
  if [ ! -x $PY ]; then
    $MAMBA create -p $CENV $CH python=3.8 pytorch=1.9.0 cudatoolkit=10.2 pytorch-lightning=1.3.8 ase=3.22 autopep8 seaborn tqdm pip "mkl<2024.1" "torchmetrics<0.7" "setuptools<=59.5.0"
  fi
  $MAMBA install -p $CENV $CH matminer=0.7.3 pymatgen=2020.12.31 "numpy<1.24" "mkl<2024.1" "ruamel.yaml<0.18" "torchmetrics<0.7" "monty<2022" "setuptools<=59.5.0"
  $PIP install --no-index -f https://data.pyg.org/whl/torch-1.9.0+cu102.html torch-scatter==2.0.9 torch-sparse==0.6.12 torch-cluster==1.5.9 torch-spline-conv==1.2.1
  $PIP install torch-geometric==1.7.2 higher==0.2.1 hydra-core==1.1.0 hydra-joblib-launcher==1.1.5 p-tqdm==1.3.3 pytest python-dotenv smact==2.2.1 streamlit==0.79.0 torchdiffeq==0.0.1 wandb==0.10.33
  $PIP install -q "protobuf<4" -e $ROOT/cdvae
fi

echo "== 3. final check =="
ROOT="$ROOT" $PY - <<'EOF'
import os, sys, importlib, importlib.metadata as m, torch
ROOT = os.environ.get("ROOT", "/content")
for k, v in {"PROJECT_ROOT": ROOT + "/cdvae", "HYDRA_JOBS": "/tmp/hydra_check", "WABDB_DIR": "/tmp/wandb_check"}.items():
    os.environ.setdefault(k, v)
sys.path.insert(0, ROOT + "/cdvae/scripts")
ok = torch.cuda.is_available()
print("torch", torch.__version__, "| GPU ok:", ok)
if ok:
    try:
        print("GPU:", torch.cuda.get_device_name(0), "| tiny test:", (torch.ones(2, device="cuda") * 2).sum().item())
    except Exception as e:
        print("GPU found but this old PyTorch cannot use it:", e)
else:
    print("NOTE: this machine has NO GPU (fine for preparing data, not for training)")
bad = []
for mod in ["torch_scatter", "torch_sparse", "torch_cluster", "torch_geometric.data", "torch_geometric.nn.acts",
            "pymatgen.core.structure", "pymatgen.analysis.graphs", "pymatgen.analysis.local_env",
            "pymatgen.analysis.structure_matcher", "matminer.featurizers.site.fingerprint",
            "matminer.featurizers.composition.composite", "smact.screening", "wandb", "p_tqdm",
            "cdvae.common.data_utils", "cdvae.pl_data.datamodule", "cdvae.pl_modules.model", "cdvae.run", "eval_utils"]:
    try:
        importlib.import_module(mod)
    except Exception as e:
        bad.append(f"{mod}: {type(e).__name__}: {str(e)[:200]}")
print("ALL CDVAE IMPORTS OK" if not bad else "IMPORT PROBLEMS:\n  " + "\n  ".join(bad))
sys.exit(1 if bad else 0)
EOF

echo "== 4. save the environment to Drive (once) so a reset costs minutes, not 25 =="
if [ -d $DRIVE ] && [ ! -f $ARCHIVE ]; then
  ( tar -I "gzip -1" -cf $ARCHIVE.part -C $(dirname $ROOT) $(basename $ROOT)/cdvae_env && mv $ARCHIVE.part $ARCHIVE && ls -lh $ARCHIVE ) \
    || { rm -f $ARCHIVE.part; echo "could not save to Drive (not enough space?) - not fatal"; }
else
  echo "already saved (or Drive not mounted)"
fi

== 0. repo (skipped if already there) ==
== 1. restore the environment saved on Drive, if there is one ==
restoring /content/drive/MyDrive/cdvae_env_backup.tar.gz (a few minutes) ...
== 2. environment is already complete: skipping all installs ==
== 3. final check ==
torch 1.9.0 | GPU ok: True
GPU: Tesla T4 | tiny test: 4.0
ALL CDVAE IMPORTS OK
== 4. save the environment to Drive (once) so a reset costs minutes, not 25 ==
already saved (or Drive not mounted)


Cell 5: mini test (about 3 min; works on CPU or GPU)

In [ ]:
import csv, os, subprocess, time
csv.field_size_limit(10**9)
src, dst = "/content/cdvae/data/perov_5_grouped", "/content/cdvae/data/perov_5_mini"
os.makedirs(dst, exist_ok=True)
for name, n in [("train", 512), ("val", 128), ("test", 128)]:
    with open(f"{src}/{name}.csv", newline="") as f, open(f"{dst}/{name}.csv", "w", newline="") as g:
        r, w = csv.reader(f), csv.writer(g)
        w.writerow(next(r))
        for i, row in enumerate(r):
            if i >= n:
                break
            w.writerow(row)
has_gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True).returncode == 0
DEVICE = "" if has_gpu else "train.pl_trainer.gpus=0"
print("GPU found: testing on the GPU" if has_gpu else "NO GPU: testing on the CPU (slower, but checks the whole pipeline)")
t0 = time.time()
!rm -rf {RUNS}/smoke
!cd /content/cdvae && {ENV} {PY} cdvae/run.py data=perov model.predict_property=True data.root_path={dst} data.preprocess_workers=2 logging.wandb.mode=offline expname=smoke hydra.run.dir={RUNS}/smoke data.train_max_epochs=3 data.teacher_forcing_max_epoch=2 logging.val_check_interval=1 {DEVICE}
print(f"\nmini test took {(time.time() - t0) / 60:.1f} min in total")

GPU found: testing on the GPU
[2026-11-02 16:32:34,389][hydra.utils][INFO] - Instantiating <cdvae.pl_data.datamodule.CrystDataModule>
[cache] loading converted data from /content/drive/MyDrive/cdvae_cache/train_a2541294d2622118.pkl
/content/cdvae/cdvae/common/data_utils.py:612: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
[2026-11-02 16:32:37,197][hydra.utils][INFO] - Instantiating <cdvae.pl_modules.model.CDVAE>
[2026-11-02 16:32:50,302][hydra.utils][INFO] - Passing scaler from datamodule to model <StandardScalerTorch(means: 1.3787975311279297, stds: 0.6557778716087341)>
[2026-11-02 16:32:50,318][hydra.utils][INFO] - Adding callback <LearningRateMonitor>
[2026-11-02 16:32:50,318][hydra.utils][INFO] - Adding callback <EarlyStopping>
[2026-11-02 16:32:50,320][hydra.utils][INFO] - Adding callback <Mo

Cell 6: training

In [ ]:
import time
EPOCHS = 600   # set this after Cell 5 on the GPU: hours ~ EPOCHS x 23 x (seconds per batch) / 3600
t0 = time.time()
!cd /content/cdvae && {ENV} {PY} cdvae/run.py {COMMON} expname=perov_fair hydra.run.dir={RUNS}/perov_fair data.train_max_epochs={EPOCHS} data.teacher_forcing_max_epoch={EPOCHS//2}
print(f"\nthis session trained for {(time.time() - t0) / 3600:.1f} h")

[2026-11-02 16:33:07,801][hydra.utils][INFO] - Instantiating <cdvae.pl_data.datamodule.CrystDataModule>
[cache] loading converted data from /content/drive/MyDrive/cdvae_cache/train_2e86ec062b07bf5b.pkl
/content/cdvae/cdvae/common/data_utils.py:612: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
[2026-11-02 16:33:11,621][hydra.utils][INFO] - Instantiating <cdvae.pl_modules.model.CDVAE>
[2026-11-02 16:33:25,857][hydra.utils][INFO] - Passing scaler from datamodule to model <StandardScalerTorch(means: 1.4011187553405762, stds: 0.6832627654075623)>
[2026-11-02 16:33:27,007][hydra.utils][INFO] - Adding callback <LearningRateMonitor>
[2026-11-02 16:33:27,007][hydra.utils][INFO] - Adding callback <EarlyStopping>
[2026-11-02 16:33:27,009][hydra.utils][INFO] - Adding callback <ModelCheckpoint>
/content/cdvae_

Cell 7: generate (after training). The first run is a 1,000-sample test; change 2 to 20 for 10,000 samples.

In [ ]:
!cd /content/cdvae && {ENV} {PY} scripts/evaluate.py --model_path {RUNS}/perov_fair --tasks gen opt --start_from random --num_batches_to_samples 2 --batch_size 500

/content/cdvae_env/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(
Evaluate model on the generation task.
/content/cdvae/cdvae/common/data_utils.py:622: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
  0% 0/50 [00:00<?, ?it/s]/content/cdvae_env/lib/python3.8/site-packages/torch/_tensor.py:575: UserWarning: floor_divide is deprecated, and will be removed in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values.
To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor'). (Triggered internally at  /o

In [ ]:
   import glob, os
   ck = sorted(glob.glob(f"{RUNS}/perov_fair/*.ckpt"))
   print("training checkpoint(s) on Drive:", [os.path.basename(c) for c in ck] or "none")

training checkpoint(s) on Drive: ['epoch=379-step=8739.ckpt', 'epoch=594-step=13684.ckpt']


Cell 8: write the scorer file

In [ ]:
%%writefile step4_score.py
"""Step 4 - score what CDVAE generated against the menu.
Run inside the CDVAE environment:  python step4_score.py <model_dir>/eval_gen.pt <model_dir>/eval_opt.pt
"""
import collections, csv, itertools, os, sys
import numpy as np

MENU = "cdvae/data/perov_5_grouped/menu.csv"
TOP_FRACTION = 0.10      # same definition of "good" as the loop
START_POINTS = 100       # CDVAE optimises 100 trajectories and stacks the snapshots in blocks of 100
ANIONS = {"O", "N", "F", "S"}
SYMBOLS = ("H He Li Be B C N O F Ne Na Mg Al Si P S Cl Ar K Ca Sc Ti V Cr Mn Fe Co Ni Cu Zn Ga Ge As Se Br Kr "
           "Rb Sr Y Zr Nb Mo Tc Ru Rh Pd Ag Cd In Sn Sb Te I Xe Cs Ba La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb Lu "
           "Hf Ta W Re Os Ir Pt Au Hg Tl Pb Bi Po At Rn Fr Ra Ac Th Pa U Np Pu Am Cm Bk Cf Es Fm Md No Lr").split()
SHIFTS = np.array(list(itertools.product([-1, 0, 1], repeat=3)))
# EU Critical Raw Materials Act (Regulation 2024/1252, Annex II) as chemical elements (see the sustainability cell)
EU_LIST = {"Sb", "As", "Al", "Ba", "Be", "Bi", "B", "Co", "Cu", "Ga", "Ge", "Hf", "Li", "Mg", "Mn", "Ni", "Nb", "Sc", "Si",
           "Sr", "Ta", "Ti", "W", "V", "P", "Dy", "Er", "Eu", "Gd", "Ho", "Lu", "Tb", "Tm", "Yb", "Y", "Ce", "La", "Nd",
           "Pr", "Sm", "Ir", "Pd", "Pt", "Rh", "Ru"}


def load_generated(path):
    if path.endswith(".npz"):
        return dict(np.load(path))
    import torch
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:                                   # old torch has no weights_only argument
        return torch.load(path, map_location="cpu")


def unpack(d):
    """Cut the stacked arrays into one (frac_coords, atom_numbers, lengths, angles) tuple per crystal."""
    fc, at, ln, an, na = (np.asarray(d[k]) for k in ["frac_coords", "atom_types", "lengths", "angles", "num_atoms"])
    out = []
    for e in range(fc.shape[0]):
        start = 0
        for m, n in enumerate(na[e].astype(int)):
            out.append((fc[e, start:start + n], at[e, start:start + n].astype(int), ln[e, m], an[e, m]))
            start += n
    return out


def lattice_matrix(lengths, angles):
    a, b, c = lengths
    al, be, ga = np.radians(angles)
    bx, by = b * np.cos(ga), b * np.sin(ga)
    cx = c * np.cos(be)
    cy = c * (np.cos(al) - np.cos(be) * np.cos(ga)) / np.sin(ga)
    cz = np.sqrt(max(c * c - cx * cx - cy * cy, 0.0))
    return np.array([[a, 0, 0], [bx, by, 0], [cx, cy, cz]])


def near(p, q, L):
    """Shortest periodic distance (Angstrom) between two fractional points."""
    d = p - q
    d = d - np.round(d)
    return np.linalg.norm((d + SHIFTS) @ L, axis=1).min()


def classify(frac, Z, lengths, angles, menu, metals, anion_sets):
    syms = [SYMBOLS[z - 1] if 1 <= z <= len(SYMBOLS) else "?" for z in Z]
    formula = "".join(f"{s}{c}" for s, c in sorted(collections.Counter(syms).items()))
    if len(syms) != 5:
        return "wrong number of atoms", None, None, formula
    L = lattice_matrix(lengths, angles)
    dmin = min(near(frac[i], frac[j], L) for i, j in itertools.combinations(range(5), 2))
    an = [i for i, s in enumerate(syms) if s in ANIONS]
    ca = [i for i, s in enumerate(syms) if s not in ANIONS]
    if len(an) != 3 or len(ca) != 2:
        return "not 2 metals + 3 anions", None, dmin, formula
    closest = [min(near(frac[c], frac[x], L) for x in an) for c in ca]
    b = int(np.argmin(closest))                         # B site = the metal closest to the anions
    A, B = syms[ca[1 - b]], syms[ca[b]]
    X = "".join(sorted(syms[x] for x in an))
    if A not in metals or B not in metals:
        return "element outside the menu", None, dmin, formula
    if X not in anion_sets:
        return "anion mix not among the 7", None, dmin, formula
    return "on the menu", (A, B, X), dmin, formula


def report(title, samples, ctx, seconds=None):
    n = len(samples)
    menu, thr = ctx["menu"], ctx["thr"]
    print(f"\n=== {title}: {n} samples" + (f" | sampling took {seconds / 3600:.2f} h" if seconds else "") + " ===")
    for k, v in collections.Counter(s[0] for s in samples).most_common():
        print(f"  {k:<28s}{v:>6d}  ({100 * v / n:.1f}%)")
    on = [s for s in samples if s[0] == "on the menu"]
    if on:
        heat = np.array([menu[s[1]][1] for s in on])
        held = np.array([menu[s[1]][0] != "train" for s in on])
        hit = heat <= thr
        print(f"  --- the {len(on)} on-menu samples ---")
        print(f"  seen in training          : {100 * (~held).mean():5.1f}%  (pure memorising would be 100%)")
        print(f"  never seen (held-out)     : {100 * held.mean():5.1f}%  (picking blindly from the menu gives ~{100 * ctx['held_share']:.0f}%)")
        print(f"  distinct menu entries     : {len({s[1] for s in on})} | distinct held-out: "
              f"{len({s[1] for s in on if menu[s[1]][0] != 'train'})} of {ctx['n_held']}")
        se = np.sqrt(hit.mean() * (1 - hit.mean()) / len(hit))
        print(f"  hit rate (best {TOP_FRACTION:.0%})      : {100 * hit.mean():5.1f}% +/- {100 * se:.1f}  (chance {TOP_FRACTION:.0%} -> x{hit.mean() / TOP_FRACTION:.1f})")
        if held.any():
            print(f"  hit rate, held-out only   : {100 * hit[held].mean():5.1f}%")
        free = np.array([not (s[1][0] in EU_LIST or s[1][1] in EU_LIST) for s in on])
        print(f"  free of EU-list metals    : {100 * free.mean():5.1f}%  (whole menu: {100 * ctx['free_share']:.0f}%)")
        if free.any():
            print(f"  hit rate among those      : {100 * hit[free].mean():5.1f}%  = {int(hit[free].sum())} of {int(free.sum())}  (whole menu: {100 * ctx['free_hit']:.1f}%)")
        print(f"  median heat_ref           : {np.median(heat):.2f}  (whole menu {ctx['median']:.2f})")
        print(f"  atoms closer than 1 A     : {100 * np.mean([s[2] < 1.0 for s in on]):.1f}%")
    off = collections.Counter((s[3], s[0]) for s in samples if s[0] not in ("on the menu", "wrong number of atoms"))
    if off:
        print("  most common off-menu formulas (judge them with your own chemistry):")
        for (f, why), v in off.most_common(10):
            print(f"    {f:<14s} x{v:<5d} {why}")


def snapshot_table(name, blocks, ctx):
    print(f"\n=== {name}: what the optimiser does, snapshot by snapshot (100 samples each) ===")
    print("  snapshot | on the menu | usable* | hit rate of usable | usable hits / 100 | median heat_ref of usable")
    for k, b in enumerate(blocks, 1):
        on = [s for s in b if s[0] == "on the menu"]
        use = [s for s in on if s[2] is not None and s[2] >= 1.0]
        heat = np.array([ctx["menu"][s[1]][1] for s in use])
        hits = int((heat <= ctx["thr"]).sum()) if len(use) else 0
        rate = f"{100 * hits / len(use):5.1f}%" if len(use) else "   n/a"
        med = f"{np.median(heat):.2f}" if len(use) else " n/a"
        print(f"  {k:>8d} | {100 * len(on) / len(b):10.0f}% | {100 * len(use) / len(b):6.0f}% | {rate:>18s} | {hits:>17d} | {med:>25s}")
    print("  * usable = on the menu AND no two atoms closer than 1 Angstrom.  Blind picking from the menu gives ~10 hits / 100.")


def main(paths):
    menu = {}
    with open(MENU, newline="") as f:
        for r in csv.DictReader(f):
            menu[(r["A"], r["B"], r["anions"])] = (r["split_fair"], float(r["heat_ref"]))
    metals = {k[0] for k in menu} | {k[1] for k in menu}
    anion_sets = {k[2] for k in menu}
    heats = np.array([v[1] for v in menu.values()])
    n_held = sum(v[0] != "train" for v in menu.values())
    ctx = dict(menu=menu, thr=float(np.quantile(heats, TOP_FRACTION)), n_held=n_held,
               held_share=n_held / len(menu), median=float(np.median(heats)))
    free_keys = [k for k in menu if k[0] not in EU_LIST and k[1] not in EU_LIST]
    ctx["free_share"] = len(free_keys) / len(menu)
    ctx["free_hit"] = float(np.mean([menu[k][1] <= ctx["thr"] for k in free_keys]))
    print(f"menu: {len(menu)} combinations | good = heat_ref <= {ctx['thr']:.2f} | held-out by the fair split: {n_held}")
    for path in paths:
        d = load_generated(path)
        samples = [classify(*c, menu, metals, anion_sets) for c in unpack(d)]
        name = os.path.basename(path)
        secs = float(d["time"]) if "time" in d else None
        if "opt" in name:                               # snapshots are stacked in blocks of START_POINTS
            blocks = [samples[i:i + START_POINTS] for i in range(0, len(samples), START_POINTS)]
            report(f"{name} BEFORE optimisation (first snapshot)", blocks[0], ctx)
            report(f"{name} AFTER optimisation (last snapshot)", blocks[-1], ctx, secs)
            snapshot_table(name, blocks, ctx)
        else:
            report(name, samples, ctx, secs)
    print("\nthe loop for comparison: hit rate 100% (it reads the table), milliseconds, no GPU.")


if __name__ == "__main__":
    main(sys.argv[1:])

Overwriting step4_score.py


In [ ]:
   !python step4_score.py {RUNS}/perov_fair/eval_gen.pt {RUNS}/perov_fair/eval_opt.pt

menu: 18928 combinations | good = heat_ref <= 0.61 | held-out by the fair split: 7570

=== eval_gen.pt: 1000 samples | sampling took 0.77 h ===
  on the menu                    992  (99.2%)
  anion mix not among the 7        8  (0.8%)
  --- the 992 on-menu samples ---
  seen in training          :  60.1%  (pure memorising would be 100%)
  never seen (held-out)     :  39.9%  (picking blindly from the menu gives ~40%)
  distinct menu entries     : 957 | distinct held-out: 378 of 7570
  hit rate (best 10%)      :   7.7% +/- 0.8  (chance 10% -> x0.8)
  hit rate, held-out only   :   6.8%
  free of EU-list metals    :  17.2%  (whole menu: 16%)
  hit rate among those      :  13.5%  = 23 of 171  (whole menu: 9.5%)
  median heat_ref           : 1.27  (whole menu 1.33)
  atoms closer than 1 A     : 0.0%
  most common off-menu formulas (judge them with your own chemistry):
    Cs1Hg1N1O1S1   x1     anion mix not among the 7
    Ba1Co1F1N2     x1     anion mix not among the 7
    Ba1F1Ge1N2     x1

Two optional extras. Put them at the bottom of the notebook.

Extra A: the simple loop (your baseline, 1 second)

In [ ]:
"""Step 2 - the simple loop: go through every combination on the menu and keep the good ones."""
import csv, time

MENU = "cdvae/data/perov_5_grouped/menu.csv"
TOP_FRACTION = 0.10          # "good" = the best 10% of the menu (lowest heat_ref)
AVOID = {"Co"}               # sustainability filter: replace with YOUR published list (e.g. a critical-raw-materials list)

with open(MENU, newline="") as f:
    menu = list(csv.DictReader(f))
heats = sorted(float(r["heat_ref"]) for r in menu)
threshold = heats[int(TOP_FRACTION * len(heats))]
print(f"menu: {len(menu)} combinations | 'good' means heat_ref <= {threshold:.2f} (best {TOP_FRACTION:.0%})")

t0 = time.time()
good, good_clean = [], []
for r in menu:                                    # <- this is the whole "AI-free" method
    if float(r["heat_ref"]) <= threshold:
        good.append(r)
        if r["A"] not in AVOID and r["B"] not in AVOID:
            good_clean.append(r)
seconds = time.time() - t0

print(f"loop found {len(good)} good combinations in {seconds:.3f} s")
print(f"after avoiding {sorted(AVOID)}: {len(good_clean)} left (removed {len(good) - len(good_clean)})")
print("\nbest 10 after the filter:")
for r in sorted(good_clean, key=lambda r: float(r["heat_ref"]))[:10]:
    print(f"  A={r['A']:<3s} B={r['B']:<3s} anions={r['anions']:<4s} heat_ref={float(r['heat_ref']):.2f}  (fair split: {r['split_fair']})")
print("\nrandom picking from the menu would hit 'good' about 10% of the time; the loop hits 100%.")

menu: 18928 combinations | 'good' means heat_ref <= 0.61 (best 10%)
loop found 1893 good combinations in 0.010 s
after avoiding ['Co']: 1868 left (removed 25)

best 10 after the filter:
  A=Ba  B=Nb  anions=OOO  heat_ref=-0.35  (fair split: train)
  A=K   B=W   anions=OOO  heat_ref=-0.33  (fair split: train)
  A=Ba  B=V   anions=OOO  heat_ref=-0.29  (fair split: train)
  A=Ba  B=Zr  anions=OOO  heat_ref=-0.28  (fair split: test)
  A=Sr  B=V   anions=OOO  heat_ref=-0.27  (fair split: train)
  A=Ba  B=Hf  anions=OOO  heat_ref=-0.27  (fair split: test)
  A=Ba  B=Ti  anions=OOO  heat_ref=-0.26  (fair split: train)
  A=Na  B=W   anions=OOO  heat_ref=-0.25  (fair split: val)
  A=Rb  B=W   anions=OOO  heat_ref=-0.24  (fair split: train)
  A=Sr  B=Nb  anions=OOO  heat_ref=-0.24  (fair split: train)

random picking from the menu would hit 'good' about 10% of the time; the loop hits 100%.


Extra B: warm-up. You only need this if the cdvae_cache folder on Drive is ever deleted. It rebuilds the data cache in about 26 minutes.

In [ ]:
code = '''
import os, sys, time
import hydra
from hydra import compose, initialize_config_dir

t0 = time.time()
with initialize_config_dir(config_dir="/content/cdvae/conf"):
    cfg = compose(config_name="default", overrides=sys.argv[1:])
datamodule = hydra.utils.instantiate(cfg.data.datamodule, _recursive_=False)
datamodule.setup("fit")
datamodule.setup("test")
print()
print("all three data sets are converted and saved: %.1f min" % ((time.time() - t0) / 60))
print(sorted(os.listdir(os.environ["CDVAE_CACHE"])))
'''
open("/content/warm_cache.py", "w").write(code)
!cd /content/cdvae && {ENV} {PY} /content/warm_cache.py {COMMON} expname=warm

[cache] loading converted data from /content/drive/MyDrive/cdvae_cache/train_2e86ec062b07bf5b.pkl
/content/cdvae/cdvae/common/data_utils.py:612: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
[cache] loading converted data from /content/drive/MyDrive/cdvae_cache/train_2e86ec062b07bf5b.pkl
[cache] loading converted data from /content/drive/MyDrive/cdvae_cache/val_f4021912dc998979.pkl
[cache] loading converted data from /content/drive/MyDrive/cdvae_cache/test_32f5a75eaef57038.pkl

all three data sets are converted and saved: 0.1 min
['test_32f5a75eaef57038.pkl', 'test_8bfcc7394dc20efd.pkl', 'train_2e86ec062b07bf5b.pkl', 'train_a2541294d2622118.pkl', 'val_aed1a53882705ed1.pkl', 'val_f4021912dc998979.pkl']


Appendix C

In [3]:
import csv

MENU = "cdvae/data/perov_5_grouped/menu.csv"
TOP_FRACTION = 0.10

# EU Critical Raw Materials Act (Regulation 2024/1252, Annex II) written as chemical elements.
# Rare earths = the 15 REE of the EU study (HREE: Dy Er Eu Gd Ho Lu Tb Tm Yb Y; LREE: Ce La Nd Pr Sm) + scandium.
# Platinum-group metals = Ir Pd Pt Rh Ru.  Fluorine is flagged separately (the listed raw material is "fluorspar").
EU_LIST = {"Sb", "As", "Al", "Ba", "Be", "Bi", "B", "Co", "Cu", "Ga", "Ge", "Hf", "Li", "Mg", "Mn", "Ni", "Nb", "Sc", "Si",
           "Sr", "Ta", "Ti", "W", "V", "P", "Dy", "Er", "Eu", "Gd", "Ho", "Lu", "Tb", "Tm", "Yb", "Y", "Ce", "La", "Nd",
           "Pr", "Sm", "Ir", "Pd", "Pt", "Rh", "Ru"}

with open(MENU, newline="") as f:
    menu = list(csv.DictReader(f))
heats = sorted(float(r["heat_ref"]) for r in menu)
threshold = heats[int(TOP_FRACTION * len(heats))]
good = [r for r in menu if float(r["heat_ref"]) <= threshold]

metal_free = lambda r: r["A"] not in EU_LIST and r["B"] not in EU_LIST
fully_free = lambda r: metal_free(r) and "F" not in r["anions"]

print(f"menu {len(menu)} | good = best {TOP_FRACTION:.0%} (heat_ref <= {threshold:.2f}): {len(good)}")
print(f"avoid Co only                    : {sum(r['A'] != 'Co' and r['B'] != 'Co' for r in good)} good combos left")
print(f"avoid EU-list metals             : {sum(metal_free(r) for r in good)} left")
print(f"avoid EU-list metals + fluorine  : {sum(fully_free(r) for r in good)} left")
print(f"whole menu free of EU-list metals: {sum(metal_free(r) for r in menu)} of {len(menu)}")
print("\nbest 10 without any EU-list metal:")
for r in sorted(filter(metal_free, menu), key=lambda r: float(r["heat_ref"]))[:10]:
    print(f"  A={r['A']:<3s} B={r['B']:<3s} anions={r['anions']:<4s} heat_ref={float(r['heat_ref']):.2f}  (fair split: {r['split_fair']})")

menu 18928 | good = best 10% (heat_ref <= 0.61): 1893
avoid Co only                    : 1868 good combos left
avoid EU-list metals             : 293 left
avoid EU-list metals + fluorine  : 181 left
whole menu free of EU-list metals: 3087 of 18928

best 10 without any EU-list metal:
  A=K   B=Mo  anions=OOO  heat_ref=-0.22  (fair split: val)
  A=Na  B=Mo  anions=OOO  heat_ref=-0.14  (fair split: val)
  A=Rb  B=Mo  anions=OOO  heat_ref=-0.14  (fair split: train)
  A=K   B=Re  anions=OOO  heat_ref=-0.05  (fair split: train)
  A=Ca  B=Cr  anions=OOO  heat_ref=-0.04  (fair split: val)
  A=Na  B=Re  anions=OOO  heat_ref=-0.04  (fair split: test)
  A=Ca  B=Zr  anions=OOO  heat_ref=0.02  (fair split: test)
  A=Ca  B=Cr  anions=FOO  heat_ref=0.05  (fair split: train)
  A=Ca  B=Re  anions=NNN  heat_ref=0.05  (fair split: train)
  A=Cs  B=Mo  anions=OOO  heat_ref=0.05  (fair split: val)
